<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/golf/02_angle_geometry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Golf putting 2 — Geometry: angle error

Replace the generic logistic curve with a one-parameter model derived from the geometry of a ball entering a cup. Fit it to the Berry data, then test the already-fit model on the later Broadie data **without refitting**.

## Setup

The notebook pins the current PyMC 6 / ArviZ 1.x stack used in the course.

In [ ]:
%pip install -q "pymc==6.3.2" "arviz>=1.3,<2"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260923
azp.style.use("arviz-variat")

DATA_BASE = "https://raw.githubusercontent.com/opherdonchin/BayesShortCourse/main/golf/data"
golf = pd.read_csv(f"{DATA_BASE}/berry_1996_putting.csv")
golf["rate"] = golf["made"] / golf["attempts"]

BALL_RADIUS_FT = (1.68 / 2) / 12
CUP_RADIUS_FT = (4.25 / 2) / 12

golf

In [ ]:
def plot_data(data, title=None):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(data["distance_ft"], data["rate"], s=35)
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Proportion made",
        ylim=(-0.03, 1.03),
        title=title,
    )
    return ax

def plot_predictive_rate(dt, group, data, var_name, title, prob=0.90, show_observed=True):
    draws = dt[group][var_name]
    median = draws.median(dim=("chain", "draw"))
    interval = draws.azstats.hdi(prob=prob)
    x = data["distance_ft"].to_numpy()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.fill_between(
        x,
        interval.sel(ci_bound="lower"),
        interval.sel(ci_bound="upper"),
        alpha=0.22,
        label=f"{prob:.0%} HDI",
    )
    ax.plot(x, median, label="Predictive median")
    if show_observed:
        ax.scatter(x, data["rate"], s=35, label="Observed")
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Proportion made",
        ylim=(-0.05, 1.05),
        title=title,
    )
    ax.legend()
    return ax

def plot_latent_fit(idata, data, var_name="p_base", title="Underlying fitted relationship"):
    p = idata["posterior"][var_name]
    median = p.median(dim=("chain", "draw"))
    interval = p.azstats.hdi(prob=0.90)
    x = data["distance_ft"].to_numpy()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.fill_between(
        x,
        interval.sel(ci_bound="lower"),
        interval.sel(ci_bound="upper"),
        alpha=0.22,
        label="90% HDI",
    )
    ax.plot(x, median, label="Posterior median")
    ax.scatter(x, data["rate"], s=35, label="Observed")
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Proportion made",
        ylim=(-0.03, 1.03),
        title=title,
    )
    ax.legend()
    return ax

def plot_residuals(idata, data, var_name="p_base", title="Residuals"):
    fitted = idata["posterior"][var_name].median(dim=("chain", "draw"))
    residual = data["rate"].to_numpy() - fitted.to_numpy()
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.plot(data["distance_ft"], residual, marker="o")
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Observed − fitted probability",
        title=title,
    )
    return ax

## Model from first principles

If a golfer aims straight but the actual direction has Normal angular error with standard deviation \(\sigma_{angle}\), a putt from distance \(x\) succeeds when the angular error stays within

\[
\theta(x)=\arcsin\!\left(\frac{R-r}{x}\right).
\]

Thus

\[
p(x)=2\Phi\!\left(\frac{\theta(x)}{\sigma_{angle}}\right)-1.
\]

We put the prior directly on \(\sigma_{angle}\) in **degrees**.

In [ ]:
coords = {"obs_id": golf["distance_ft"].to_numpy()}

with pm.Model(coords=coords) as model:
    distance = pm.Data("distance", golf["distance_ft"].to_numpy(), dims="obs_id")
    attempts = pm.Data("attempts", golf["attempts"].to_numpy(), dims="obs_id")
    made_data = pm.Data("made_data", golf["made"].to_numpy(), dims="obs_id")

    sigma_angle_deg = pm.LogNormal("sigma_angle_deg", mu=np.log(2), sigma=0.7)
    sigma_angle_rad = sigma_angle_deg * np.pi / 180
    threshold_angle = pm.math.arcsin((CUP_RADIUS_FT - BALL_RADIUS_FT) / distance)
    p_base = pm.Deterministic(
        "p_base",
        2 * pm.math.invprobit(threshold_angle / sigma_angle_rad) - 1,
        dims="obs_id",
    )
    p = pm.Deterministic("p", p_base, dims="obs_id")

    made = pm.Binomial(
        "made",
        n=attempts,
        p=p,
        observed=made_data,
        dims="obs_id",
    )
    pm.Deterministic("made_rate", made / attempts, dims="obs_id")

## Prior predictive check

Check what the model can generate **before conditioning on the observed successes**. The design variables (distance and number of attempts) are fixed; the outcomes are simulated.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=500, random_seed=RANDOM_SEED)

In [ ]:
plot_predictive_rate(prior, "prior_predictive", golf, "made_rate", "Prior predictive", show_observed=False);

## Fit and diagnose

Do not interpret the scientific fit until the sampler diagnostics are acceptable.

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        target_accept=0.9,
        nuts_sampler="pymc",
        random_seed=RANDOM_SEED,
    )

print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))
azs.summary(
    idata,
    var_names=['sigma_angle_deg'],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(idata, var_names=['sigma_angle_deg']);

## Posterior predictive check

In [ ]:
with model:
    pm.sample_posterior_predictive(
        idata,
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

In [ ]:
plot_predictive_rate(idata, "posterior_predictive", golf, "made_rate", "Posterior predictive check");

In [ ]:
plot_latent_fit(idata, golf, var_name="p_base", title="Angle-only geometry model");
plot_residuals(idata, golf, var_name="p_base", title="Angle-model residuals on Berry data");

## External check on newer data

Now use the posterior learned from the Berry data to predict the Broadie observations. We do **not** condition on the new outcomes.

In [ ]:
new_golf = pd.read_csv(f"{DATA_BASE}/broadie_2018_putting.csv")
new_golf["rate"] = new_golf["made"] / new_golf["attempts"]

new_coords = {"obs_id": new_golf["distance_ft"].to_numpy()}
with pm.Model(coords=new_coords) as prediction_model:
    distance = pm.Data("distance_new", new_golf["distance_ft"].to_numpy(), dims="obs_id")
    attempts = pm.Data("attempts_new", new_golf["attempts"].to_numpy(), dims="obs_id")
    sigma_angle_deg = pm.LogNormal("sigma_angle_deg", mu=np.log(2), sigma=0.7)
    sigma_angle_rad = sigma_angle_deg * np.pi / 180
    threshold_angle = pm.math.arcsin((CUP_RADIUS_FT - BALL_RADIUS_FT) / distance)
    p_new = pm.Deterministic(
        "p_new",
        2 * pm.math.invprobit(threshold_angle / sigma_angle_rad) - 1,
        dims="obs_id",
    )
    made_new = pm.Binomial("made_new", n=attempts, p=p_new, dims="obs_id")
    pm.Deterministic("made_rate_new", made_new / attempts, dims="obs_id")

    predictions = pm.sample_posterior_predictive(
        idata,
        predictions=True,
        var_names=["p_new", "made_rate_new"],
        freeze_vars=["sigma_angle_deg"],
        random_seed=RANDOM_SEED,
    )

In [ ]:
plot_predictive_rate(predictions, "predictions", new_golf, "made_rate_new", "Already-fit angle model on new Broadie data");

## Decision: reject for the new prediction problem

The newer data have substantially higher short-putt success and, beyond about 20 feet, lower success than the old angle-only model predicts. That is an **external predictive failure**, not merely an in-sample residual. A plausible missing mechanism is distance control: long putts can miss by being hit the wrong distance even when their angle is good. **Next notebook:** add distance error.